# 07. The Full Client-Side Function-Calling Loop

**⚠️ Dependency:** this script requires `"IT-HelpDesk-Agent"` **version 2** — the version created by
`05_IT_HelpDesk_agent.py`, which is the one carrying the three `FunctionTool` schemas. Run `02_prompt_agent.py`
then `05_IT_HelpDesk_agent.py` first, or make sure your project already has a version 2 of that agent with
function tools attached (`AGENT_VERSION = "2"` is hardcoded in the original script).

This is the payoff for `05_IT_HelpDesk_agent.py` and `helpdesk_functions.py`: it demonstrates the **complete**
client-side function-calling loop that Azure AI Foundry's `FunctionTool` requires, because — unlike
`WebSearchTool` in `04_agent_web_search.py` — the service never executes your function code itself.

The flow is:
1. Create a `conversation` (server-side conversation state container).
2. Send the user's question to the pinned agent version.
3. Inspect `response.output` for any `function_call` items — if the model wants to call a tool, execution
   *stops* here and the tool call is returned to you instead of a final answer.
4. Execute the requested function locally via `helpdesk_functions.run_local_function`.
5. Submit the function's result back as a `function_call_output` item, in the same conversation.
6. Print the final grounded answer.

**Difficulty:** Intermediate


## Prerequisites

**pip3 packages:** `azure-ai-projects`, `azure-identity`, `python-dotenv` — already in root `requirements.txt`.
Also needs `helpdesk_functions.py` importable — since it lives in the same directory, running this notebook
from that directory (or adding it to `sys.path`) is required, same as the original script.

**Azure resource(s) required:** The Foundry project with `IT-HelpDesk-Agent` version 2 already created
(via `05_IT_HelpDesk_agent.py`).

**Environment variables** (root `.env`):
- `AZURE_AI_PROJECT_ENDPOINT`
- `AZURE_AI_AGENT_NAME` (optional, defaults to `"IT-HelpDesk-Agent"`)
- `AZURE_AI_AGENT_VERSION` (optional, defaults to `"2"` to match `05_IT_HelpDesk_agent.py`'s output)


## What You'll Learn

- How `client.conversations.create()` gives you a server-side conversation container you can reuse across
  multiple `responses.create()` calls (multi-turn state, without you tracking message history yourself)
- How to detect a `function_call` output item, parse its JSON `arguments`, and dispatch it to real code
- How to submit a `function_call_output` item (keyed by `call_id`) back into the same conversation to get the
  model's final, tool-informed answer
- Why pinning `agent_reference`'s `"version"` explicitly (here, `"2"`) matters when an agent has tool-bearing
  and non-tool-bearing versions


### Setup: client, conversation, and the first request

We import `run_local_function` from the reference module covered in the `helpdesk_functions.ipynb` notebook.
`client.conversations.create()` returns a conversation object whose `.id` we pass into every subsequent
`responses.create()` call — this is what lets the second call (after executing the tool) continue the *same*
exchange rather than starting fresh. The first `responses.create()` call sends the user's VPN question,
pinned to `IT-HelpDesk-Agent` version 2 via `agent_reference`.

- **💡 Exam tip:** `conversations.create()` giving you server-side state to reuse across calls is conceptually
  the same idea as a persisted **thread** in Foundry's native Agent Service (`11_azure_ai_foundry/05_hosted_agent`)
  — both let the service remember prior turns so you don't have to resend full history yourself.
- **🔄 Alternatives:** For a single-turn, stateless question (no follow-up needed), you could skip
  `conversations.create()` entirely and call `responses.create()` without a `conversation` id, as
  `03_invoke_agent.py` and `04_agent_web_search.py` do.


In [ ]:
import json
import os

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

from helpdesk_functions import run_local_function

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
AGENT_NAME = os.getenv("AZURE_AI_AGENT_NAME", "IT-HelpDesk-Agent")
AGENT_VERSION = os.getenv("AZURE_AI_AGENT_VERSION", "2")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
client = project.get_openai_client()

conversation = client.conversations.create()

response = client.responses.create(
    conversation=conversation.id,
    input="My VPN keeps disconnecting. What should I do?",
    extra_body={
        "agent_reference": {
            "type": "agent_reference",
            "name": AGENT_NAME,
            "version": AGENT_VERSION,
        }
    },
)


### Detecting and executing tool calls

We iterate `response.output` looking for items whose `.type == "function_call"`. Each such item carries the
tool's `name`, a JSON string of `arguments`, and a `call_id` that uniquely identifies this specific tool-call
request (needed to correlate the result back correctly). `json.loads(item.arguments)` turns the JSON string
into a Python dict, which `run_local_function` then dispatches to the real implementation in
`helpdesk_functions.py`. Each result is packaged as a `{"type": "function_call_output", "call_id": ..., "output": ...}`
dict, ready to submit back to the model.

- **💡 Exam tip:** `call_id` matters — if a single turn triggers *multiple* tool calls, each result must be
  tagged with the `call_id` of the request it answers, so the model can correctly match outputs back to
  requests. This mirrors OpenAI's function-calling / tool-output correlation pattern generally.
- **🔄 Alternatives:** For built-in tools (`WebSearchTool`, code interpreter), you would **never** see a
  `function_call` item requiring this treatment — the service resolves those internally and only ever returns
  final (or intermediate built-in-tool) output, not a request for you to execute something.


In [ ]:
tool_outputs = []

for item in response.output:
    if item.type == "function_call":
        function_name = item.name
        arguments = json.loads(item.arguments)

        print(f"Function requested: {function_name}")
        print(f"Arguments received: {arguments}")

        function_result = run_local_function(function_name, arguments)

        tool_outputs.append(
            {
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": function_result,
            }
        )


### Submitting tool outputs and getting the final answer

If any tool calls were found, we make a **second** `responses.create()` call — same `conversation.id`, but
`input=tool_outputs` this time instead of a plain question. This tells the model "here are the results of the
tool(s) you asked for; now produce your final answer." If no tool calls were found at all (the model answered
directly, or a built-in tool already resolved things), we simply print the first response's `output_text`.

- **💡 Exam tip:** This two-call pattern — request, detect tool call, execute, resubmit, get final answer — is
  the canonical shape of function calling across essentially every major LLM API family (OpenAI Responses,
  OpenAI Assistants' `submit_tool_outputs`, Azure OpenAI, and here, Azure AI Foundry Agent Service). AI-102
  expects you to recognize this shape regardless of exact method names.
- **🔄 Alternatives:** Some SDKs offer a higher-level "run and auto-resolve tool calls" helper that hides this
  loop from you (the OpenAI Agents SDK's `Runner.run` in `01_openai_agent.py` is exactly that) — Azure AI
  Foundry's `FunctionTool` intentionally leaves the loop manual so you retain full control over what code
  actually executes.


In [ ]:
if tool_outputs:
    final_response = client.responses.create(
        conversation=conversation.id,
        input=tool_outputs,
        extra_body={
            "agent_reference": {
                "type": "agent_reference",
                "name": AGENT_NAME,
                "version": AGENT_VERSION,
            }
        },
    )

    print(final_response.output_text)

else:
    print(response.output_text)


## Summary

This notebook runs the complete round trip for a `FunctionTool`-equipped Foundry agent: create a conversation,
ask a VPN troubleshooting question, detect that the model wants to call `get_vpn_troubleshooting_steps`,
execute that function locally via `helpdesk_functions.run_local_function`, submit the result back into the
same conversation, and print the model's final, tool-grounded answer. This is the reference implementation for
"how do I actually use a Foundry `FunctionTool` end-to-end" — every other script in this section that uses
`FunctionTool` (`05_IT_HelpDesk_agent.py`) depends on a loop shaped exactly like this one to be usable at all.


## Try It Yourself

1. **Easy:** Change the `input` question to one that should trigger `get_software_install_guide` (e.g. "How
   do I install Zoom?") and confirm the printed `Function requested:` line matches.
2. **Intermediate:** Ask a follow-up question using the *same* `conversation.id` after this exchange completes
   (a new `responses.create()` call) and observe whether the agent remembers the prior VPN context.
3. **Advanced:** Modify the loop to handle a response containing **multiple** `function_call` items in one
   turn — build `tool_outputs` for all of them before making the single follow-up `responses.create()` call.
